# DL01 — XOR をゼロから学ぶニューラルネットワーク

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilu-h-bando/dl-tutorial/blob/main/DL01_xor.ipynb)

このノートブックは **ライブラリを一切使わず（NumPy すら使わず）**、標準ライブラリだけで
2層ニューラルネットワークを実装し、**XOR（排他的論理和）** を学習させます。

XOR は「入力が異なるとき 1、同じとき 0」を返す問題です。1本の直線では分離できない
（＝線形分離不可能な）代表例で、**隠れ層のある非線形なネットワークでないと解けません**。
だからこそ「ニューラルネットが何を学習しているのか」を体感する最初の題材に最適です。

| 入力 x1 | 入力 x2 | 出力(正解) |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**このノートで自分の手で確かめること**
- 行列の掛け算・足し算がニューラルネットの計算の正体であること
- 順伝播（forward）→ 誤差 → 逆伝播（backward）→ 更新、という学習の1サイクル
- 誤差（loss）が繰り返しとともに下がり、予測が正解に近づいていく様子

> ねらい：PyTorch や TensorFlow が内部でやっていることを、隠さずに全部見ること。

## 動かし方

- **Google Colab**：上の「Open in Colab」バッジ →「ランタイム > すべてのセルを実行」
- **手元の Jupyter**：上から順にセルを実行

追加ライブラリのインストールは不要です（標準ライブラリ `random`, `math` のみ）。
以降、セルは**上から順に**実行してください（前のセルで定義した関数を後のセルで使います）。

## 1. 準備：定数と学習データ

まずネットワークの形と学習の設定を決めます。

- `N=2`：入力の数（x1, x2）
- `O=4`：隠れ層のユニット数。ここを増やすと表現力が上がります
- `P=1`：出力の数（0〜1の値ひとつ）
- `LEARNING_RATE=0.1`：1回の更新でどれだけパラメータを動かすか（学習率）
- `EPOCHS=10000`：全データを何回繰り返し学習するか
- `random.seed(0)`：乱数を固定し、毎回同じ結果が出るようにする（再現性）

`X` が入力の4パターン、`T` がそれぞれの正解（XORの出力）です。

In [ ]:
import random
import math

N = 2
O = 4
P = 1

LEARNING_RATE = 0.1
EPOCHS = 10000

random.seed(0)

X = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]

T = [
    [0.0],
    [1.0],
    [1.0],
    [0.0],
]

## 2. 行列・ベクトル演算を自前で実装する

ニューラルネットの計算は、突き詰めると **行列の掛け算・足し算・要素ごとの掛け算** の
組み合わせです。通常は NumPy がやってくれる部分を、ここではあえて Python の二重ループで
書き下します。これを読めば「NumPy が裏で何をしているか」が分かります。

- `matmul(A, B)`：行列積（`A @ B`）
- `transpose(A)`：転置（行と列の入れ替え）。逆伝播で多用します
- `mat_add_bias`：各行にバイアスを足す
- `mat_sub` / `mat_scalar_mul` / `mat_elem_mul`：引き算・スカラー倍・要素積
- `mat_sum_axis0`：列方向の合計（バイアスの勾配に使う）
- `mat_update(W, dW, lr)`：勾配の分だけパラメータを更新（`W -= lr * dW`）
- `mean_squared_error`：予測と正解のズレ（平均二乗誤差＝損失）

In [ ]:
def matmul(A, B):
    rows = len(A)
    cols = len(B[0])
    inner = len(B)

    C = []
    for i in range(rows):
        row = []
        for j in range(cols):
            s = 0.0
            for k in range(inner):
                s += A[i][k] * B[k][j]
            row.append(s)
        C.append(row)

    return C

def transpose(A):
    rows = len(A)
    cols = len(A[0])

    Tm = []
    for j in range(cols):
        row = []
        for i in range(rows):
            row.append(A[i][j])
        Tm.append(row)

    return Tm

def mat_add_bias(A, b):
    C = []
    for i in range(len(A)):
        row = []
        for j in range(len(A[0])):
            row.append(A[i][j] + b[0][j])
        C.append(row)
    return C

def mat_sub(A, B):
    C = []
    for i in range(len(A)):
        row = []
        for j in range(len(A[0])):
            row.append(A[i][j] - B[i][j])
        C.append(row)
    return C

def mat_scalar_mul(A, s):
    C = []
    for i in range(len(A)):
        row = []
        for j in range(len(A[0])):
            row.append(A[i][j] * s)
        C.append(row)
    return C

def mat_elem_mul(A, B):
    C = []
    for i in range(len(A)):
        row = []
        for j in range(len(A[0])):
            row.append(A[i][j] * B[i][j])
        C.append(row)
    return C

def mat_sum_axis0(A):
    cols = len(A[0])
    row = []

    for j in range(cols):
        s = 0.0
        for i in range(len(A)):
            s += A[i][j]
        row.append(s)

    return [row]

def mat_update(W, dW, lr):
    for i in range(len(W)):
        for j in range(len(W[0])):
            W[i][j] -= lr * dW[i][j]

def mean_squared_error(Y, T):
    total = 0.0
    count = 0

    for i in range(len(Y)):
        for j in range(len(Y[0])):
            diff = Y[i][j] - T[i][j]
            total += diff * diff
            count += 1

    return total / count

## 3. 活性化関数

線形な計算（行列積）だけを何層重ねても、結局ひとつの線形変換にしかならず XOR は解けません。
そこで各層の出力に **非線形な関数**（活性化関数）を挟みます。これがネットワークに
「曲げる力」を与えます。

- **ReLU**：`max(0, x)`。負を 0 にするだけの単純な関数。ここでは隠れ層に使用
- **Sigmoid**：出力を 0〜1 に押し込める。ここでは最終出力に使用（確率のように読める）

逆伝播では各関数の**微分（傾き）**が必要になるので、`relu_grad` / `sigmoid_grad_from_y`
も用意します（sigmoid の微分は出力 y を使って `y*(1-y)` と簡単に書けます）。

In [ ]:
def relu(A):
    C = []
    for row in A:
        C.append([max(0.0, x) for x in row])
    return C

def relu_grad(A):
    C = []
    for row in A:
        C.append([1.0 if x > 0.0 else 0.0 for x in row])
    return C

def sigmoid(A):
    C = []
    for row in A:
        new_row = []
        for x in row:
            if x < -50:
                x = -50
            elif x > 50:
                x = 50
            new_row.append(1.0 / (1.0 + math.exp(-x)))
        C.append(new_row)
    return C

def sigmoid_grad_from_y(Y):
    C = []
    for row in Y:
        C.append([y * (1.0 - y) for y in row])
    return C

## 4. 重みの初期化

学習の出発点となる重み `W1, W2` を小さな乱数で、バイアス `b1, b2` を 0 で初期化します。

- `W1`：入力(2)→隠れ層(4) の重み、形は 2×4
- `W2`：隠れ層(4)→出力(1) の重み、形は 4×1

乱数で始めるのは、すべて同じ値だと全ユニットが同じ学習をしてしまい意味がなくなるためです
（**対称性の破れ**）。`scale=0.1` で小さめの値にしています。

In [ ]:
def rand_matrix(rows, cols, scale=0.1):
    return [
        [(random.random() * 2.0 - 1.0) * scale for _ in range(cols)]
        for _ in range(rows)
    ]

def zeros(rows, cols):
    return [[0.0 for _ in range(cols)] for _ in range(rows)]

W1 = rand_matrix(N, O, 0.1)
b1 = zeros(1, O)

W2 = rand_matrix(O, P, 0.1)
b2 = zeros(1, P)

## 5. 学習ループ（ここが心臓部）

`EPOCHS` 回、次の4ステップを繰り返します。

1. **順伝播 (forward)**：入力 → `Z1=XW1+b1` → ReLU → `Z2=HW2+b2` → Sigmoid → 予測 `Y`
2. **損失 (loss)**：`Y` と正解 `T` の平均二乗誤差を計算
3. **逆伝播 (backward)**：損失を各パラメータで微分し、勾配 `dW1, db1, dW2, db2` を求める
   （出力側から入力側へ「誤差の責任」を配分していく＝**誤差逆伝播法**）
4. **更新 (update)**：`W -= LEARNING_RATE * dW` で勾配の逆向きに少し動かす

`epoch` が進むにつれて `loss` が下がっていくのが確認できます。

In [ ]:
for epoch in range(EPOCHS):

    # ---- forward ----
    Z1 = mat_add_bias(matmul(X, W1), b1)
    H = relu(Z1)

    Z2 = mat_add_bias(matmul(H, W2), b2)
    Y = sigmoid(Z2)

    loss = mean_squared_error(Y, T)

    # ---- backward ----
    batch_size = len(X)

    dY = mat_scalar_mul(mat_sub(Y, T), 2.0 / batch_size)
    dZ2 = mat_elem_mul(dY, sigmoid_grad_from_y(Y))

    dW2 = matmul(transpose(H), dZ2)
    db2 = mat_sum_axis0(dZ2)

    dH = matmul(dZ2, transpose(W2))
    dZ1 = mat_elem_mul(dH, relu_grad(Z1))

    dW1 = matmul(transpose(X), dZ1)
    db1 = mat_sum_axis0(dZ1)

    # ---- update ----
    mat_update(W1, dW1, LEARNING_RATE)
    mat_update(b1, db1, LEARNING_RATE)
    mat_update(W2, dW2, LEARNING_RATE)
    mat_update(b2, db2, LEARNING_RATE)

    if epoch % 1000 == 0:
        print(f"epoch={epoch}, loss={loss:.6f}")

## 6. 学習結果の確認

学習後のネットワークにもう一度 `X` を入れて予測を出します。

- **予測**：0〜1 の連続値。正解 `0,1,1,0` に近い値が出れば成功
- **0/1判定**：0.5 以上を 1、未満を 0 とした最終的な答え

`[0], [1], [1], [0]` になっていれば、XOR を学習できています。

In [ ]:
Z1 = mat_add_bias(matmul(X, W1), b1)
H = relu(Z1)
Z2 = mat_add_bias(matmul(H, W2), b2)
Y = sigmoid(Z2)

print("\n入力:")
for row in X:
    print(row)

print("\n正解:")
for row in T:
    print(row)

print("\n予測:")
for row in Y:
    print([round(v, 6) for v in row])

print("\n0/1判定:")
for row in Y:
    print([1 if v >= 0.5 else 0 for v in row])

## やってみよう（実験のヒント）

数字を変えて再実行し、挙動の違いを体感してください。

- **学習率**：`LEARNING_RATE` を `0.5` や `0.01` に変えて収束の速さ・安定性を見る
- **隠れ層の広さ**：`O` を `2` に減らすと学習が難しくなるか、`8` に増やすとどうか
- **繰り返し回数**：`EPOCHS` を `1000` に減らすと精度はどうなるか
- **活性化関数**：隠れ層の `relu` を `sigmoid` に変えるとどうなるか
- **乱数**：`random.seed(0)` の `0` を別の数にすると、初期値次第で結果が変わることを確認

> 変更したら「ランタイム > すべてのセルを実行」で最初から流し直してください。